# Delhi Public Transit Route Optimization: Exploratory Data Analysis
**Project**: Powered Public Transit Route Optimization (Infosys Springboard)  
**Dataset**: Delhi Transport Corporation (DTC) 2018 GTFS Schedule & July 2019 GPS AVL Telemetry

This notebook covers:
1. **Feasibility Assessment**: Route ID reconciliation between GPS telemetry and static GTFS schedule.
2. **Static GTFS Analysis**: Schedule row counts, missing values, service calendar, trips per route/hour, and stop mapping.
3. **GPS AVL Feed Analysis**: Readings per bus, tracking gaps, hourly speed profiling, slowest & fastest routes, and data quality / bad records audit.


## 1. Setup & Environment


In [ ]:
import os
import glob
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from folium.plugins import FastMarkerCluster

# Path resolution for local GTFS and AVL databases
def find_path(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    return None

db_path = find_path([
    '../data/raw/delhi_avl_2019/2019-07-04.db',
    '../data/raw/july_19_corrected/2019-07-04.db',
    'data/raw/delhi_avl_2019/2019-07-04.db',
    'data/raw/july_19_corrected/2019-07-04.db'
])

gtfs_dir = find_path([
    '../data/raw/delhi_gtfs_2018',
    '../data/raw',
    'data/raw/delhi_gtfs_2018',
    'data/raw'
])

print(f"Database Path: {db_path}")
print(f"GTFS Directory: {gtfs_dir}")


## 2. City Feasibility Check: GPS vs. GTFS Route ID Comparison


In [ ]:
# Check matching between GPS telemetry and 2018 timetable
conn = sqlite3.connect(db_path)
gps_routes = pd.read_sql_query("SELECT DISTINCT CAST(route_id AS INTEGER) as route_id FROM vehicle_feed WHERE route_id IS NOT NULL", conn)
conn.close()

routes_file = os.path.join(gtfs_dir, 'routes.txt')
df_routes = pd.read_csv(routes_file)

gps_route_set = set(gps_routes['route_id'])
gtfs_route_set = set(df_routes['route_id'])
matching_routes = gps_route_set.intersection(gtfs_route_set)

print(f"Total Unique GPS Route IDs: {len(gps_route_set)}")
print(f"Total Scheduled Route IDs in GTFS: {len(gtfs_route_set)}")
print(f"Matching Route IDs: {len(matching_routes)} ({len(matching_routes)/len(gps_route_set)*100:.2f}%)")
print(f"Missing in Timetable: {len(gps_route_set - gtfs_route_set)}")
print("\nDecision: 100% of GPS route IDs match the timetable -> Stay with Delhi.")


## 3. GTFS Timetable Analysis

Here we audit row counts, missing values, trip frequencies, route stop lengths, and calendar services.


In [ ]:
gtfs_files = ['agency.txt', 'calendar.txt', 'routes.txt', 'stops.txt', 'stop_times.txt', 'trips.txt']

print("=== GTFS FILE AUDIT ===")
gtfs_dfs = {}
for fname in gtfs_files:
    fpath = os.path.join(gtfs_dir, fname)
    if os.path.exists(fpath):
        df = pd.read_csv(fpath)
        gtfs_dfs[fname] = df
        null_info = df.isnull().sum()
        cols_with_null = null_info[null_info > 0]
        null_desc = ', '.join([f'{col}: {cnt} ({cnt/len(df)*100:.1f}%)' for col, cnt in cols_with_null.items()]) or 'None'
        print(f"[{fname:14s}] Rows: {len(df):7,d} | Cols: {len(df.columns):2d} | Missing: {null_desc}")


### 3.1 Trip Frequency and Hourly Distribution


In [ ]:
trips_df = gtfs_dfs['trips.txt']
routes_df = gtfs_dfs['routes.txt']
stop_times_df = gtfs_dfs['stop_times.txt']

# Trips per route
trips_per_route = trips_df.groupby('route_id').size().reset_index(name='trip_count')
trips_per_route = trips_per_route.merge(routes_df[['route_id', 'route_long_name']], on='route_id', how='left')

print("Trips per Route Summary:")
print(trips_per_route['trip_count'].describe())

# Hourly trip departure distribution at trip origin
def to_sec(s):
    p = [int(x) for x in str(s).strip().split(':')]
    return p[0]*3600 + p[1]*60 + p[2]

stop_times_df['dep_sec'] = stop_times_df['departure_time'].apply(to_sec)
trip_starts = stop_times_df.groupby('trip_id')['dep_sec'].min().reset_index()
trip_starts['start_hour'] = trip_starts['dep_sec'] // 3600
hourly_trips = trip_starts['start_hour'].value_counts().sort_index()

plt.figure(figsize=(10, 4), dpi=120)
plt.bar(hourly_trips.index, hourly_trips.values, color='#4285f4', edgecolor='black', alpha=0.8)
plt.title('Scheduled Trip Departures by Hour of Day (GTFS Timetable)', fontsize=12)
plt.xlabel('Hour of Day (24h)', fontsize=10)
plt.ylabel('Number of Departures', fontsize=10)
plt.xticks(range(4, 24))
plt.grid(axis='y', linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()


### 3.2 Route Length (Number of Stops per Route)


In [ ]:
trip_stops = stop_times_df.groupby('trip_id').size().reset_index(name='stop_count')
trip_stops = trip_stops.merge(trips_df[['trip_id', 'route_id']], on='trip_id')

route_len = trip_stops.groupby('route_id')['stop_count'].mean().reset_index(name='mean_stops')
route_len = route_len.merge(routes_df[['route_id', 'route_long_name']], on='route_id')

print("Route Length (Stops per Trip) Stats:")
print(route_len['mean_stops'].describe())

print("\nTop 5 Longest Routes:")
print(route_len.sort_values(by='mean_stops', ascending=False).head(5)[['route_id', 'route_long_name', 'mean_stops']])

print("\nTop 5 Shortest Routes:")
print(route_len.sort_values(by='mean_stops', ascending=True).head(5)[['route_id', 'route_long_name', 'mean_stops']])


### 3.3 Weekday vs. Weekend Service (calendar.txt)


In [ ]:
calendar_df = gtfs_dfs['calendar.txt']
print("calendar.txt content:")
print(calendar_df)

print("\nService ID distribution in trips.txt:")
print(trips_df['service_id'].value_counts())
print("\nObservation: All trips map to service_id=1, which runs 7 days a week. Weekday and weekend schedules are identical.")


### 3.4 Interactive Map of Stops


In [ ]:
stops_df = gtfs_dfs['stops.txt'].copy()

# Filter out bad coordinate anomalies: Stop 2546 (0,0) and identical lat/lon errors
clean_stops = stops_df[
    (stops_df['stop_lat'] > 20) & (stops_df['stop_lat'] < 35) &
    (stops_df['stop_lon'] > 70) & (stops_df['stop_lon'] < 85)
].copy()

print(f"Total stops: {len(stops_df):,} | Valid stops mapped: {len(clean_stops):,}")

m = folium.Map(location=[clean_stops['stop_lat'].mean(), clean_stops['stop_lon'].mean()], zoom_start=11, tiles='OpenStreetMap')
coords = clean_stops[['stop_lat', 'stop_lon']].values.tolist()
FastMarkerCluster(data=coords).add_to(m)

os.makedirs('maps', exist_ok=True)
m.save('maps/delhi_stops_map.html')
print("Map saved to maps/delhi_stops_map.html")


## 4. GPS Vehicle Feed Analysis (AVL Telemetry)

Investigate data quality, duplicate pings, sampling rate, reporting gaps, hourly commercial speeds, and route performance.


In [ ]:
conn = sqlite3.connect(db_path)
query = """
SELECT vehicle_id, lat, lng, speed, route_id, trip_id, CAST(timestamp AS INTEGER) as ts, date, time
FROM vehicle_feed
"""
df_gps = pd.read_sql_query(query, conn)
conn.close()

print(f"Total Raw GPS Records: {len(df_gps):,}")

# Deduplication
df_gps_clean = df_gps.drop_duplicates(subset=['vehicle_id', 'ts']).copy()
duplicates = len(df_gps) - len(df_gps_clean)
print(f"Duplicate (vehicle_id, ts) records removed: {duplicates:,} ({duplicates/len(df_gps)*100:.2f}%)")

df_gps_clean['lat'] = pd.to_numeric(df_gps_clean['lat'], errors='coerce')
df_gps_clean['lng'] = pd.to_numeric(df_gps_clean['lng'], errors='coerce')
df_gps_clean['speed'] = pd.to_numeric(df_gps_clean['speed'], errors='coerce')
df_gps_clean['route_id'] = pd.to_numeric(df_gps_clean['route_id'], errors='coerce')


### 4.1 Readings per Bus & Reporting Gaps


In [ ]:
bus_pings = df_gps_clean.groupby('vehicle_id').size()
print(f"Active buses: {len(bus_pings):,}")
print(f"Mean readings/bus/day: {bus_pings.mean():.1f} | Median: {bus_pings.median():.1f}")
print(f"Min: {bus_pings.min()} | Max: {bus_pings.max():,}")

# Gap analysis
df_sorted = df_gps_clean.sort_values(by=['vehicle_id', 'ts'])
df_sorted['prev_ts'] = df_sorted.groupby('vehicle_id')['ts'].shift(1)
df_sorted['gap_sec'] = df_sorted['ts'] - df_sorted['prev_ts']
gaps = df_sorted['gap_sec'].dropna()

print(f"Median ping interval: {gaps.median():.1f} seconds")
print(f"Intervals > 1 min:  {(gaps > 60).sum():,} ({(gaps > 60).sum()/len(gaps)*100:.2f}%)")
print(f"Intervals > 5 mins: {(gaps > 300).sum():,} ({(gaps > 300).sum()/len(gaps)*100:.2f}%)")
print(f"Intervals > 15 mins: {(gaps > 900).sum():,} ({(gaps > 900).sum()/len(gaps)*100:.2f}%)")
print(f"Intervals > 1 hour: {(gaps > 3600).sum():,} ({(gaps > 3600).sum()/len(gaps)*100:.2f}%)")


### 4.2 Average Speed by Hour of Day


In [ ]:
df_gps_clean['hour'] = df_gps_clean['time'].apply(lambda t: int(str(t).split(':')[0]) if pd.notnull(t) else None)

hourly_speed = df_gps_clean.groupby('hour')['speed'].agg(
    pings='count',
    commercial_speed='mean',
    moving_speed=lambda x: x[x > 0].mean() if (x > 0).sum() > 0 else 0
).reset_index()

hourly_speed['commercial_kmh'] = hourly_speed['commercial_speed'] * 3.6
hourly_speed['moving_kmh'] = hourly_speed['moving_speed'] * 3.6

plt.figure(figsize=(10, 5), dpi=120)
plt.plot(hourly_speed['hour'], hourly_speed['moving_kmh'], marker='o', color='#1a73e8', linewidth=2, label='Moving Speed (> 0 m/s)')
plt.plot(hourly_speed['hour'], hourly_speed['commercial_kmh'], marker='s', color='#ea4335', linewidth=2, linestyle='--', label='Commercial Speed (Overall)')
plt.title('Delhi Transit Speed Profile by Hour of Day', fontsize=12)
plt.xlabel('Hour of Day (24h)', fontsize=10)
plt.ylabel('Speed (km/h)', fontsize=10)
plt.xticks(range(0, 24))
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

hourly_speed[['hour', 'pings', 'commercial_kmh', 'moving_kmh']]


### 4.3 Slowest and Fastest Routes


In [ ]:
route_stats = df_gps_clean.groupby('route_id')['speed'].agg(
    pings='count',
    mean_speed='mean',
    moving_speed=lambda x: x[x > 0].mean() if (x > 0).sum() > 0 else 0
).reset_index()

route_stats = route_stats.merge(routes_df[['route_id', 'route_long_name']], on='route_id', how='left')
qualified_routes = route_stats[route_stats['pings'] >= 500].copy()
qualified_routes['commercial_kmh'] = qualified_routes['mean_speed'] * 3.6
qualified_routes['moving_kmh'] = qualified_routes['moving_speed'] * 3.6

print("Top 5 Fastest Routes (km/h):")
print(qualified_routes.sort_values(by='commercial_kmh', ascending=False).head(5)[['route_id', 'route_long_name', 'commercial_kmh', 'moving_kmh', 'pings']])

print("\nTop 5 Slowest Routes (km/h):")
print(qualified_routes.sort_values(by='commercial_kmh', ascending=True).head(5)[['route_id', 'route_long_name', 'commercial_kmh', 'moving_kmh', 'pings']])


## 5. Summary of Data Quality & Broken Records

1. **Duplicate Pings**:
   - `82,426` rows (**4.18%**) are exact duplicate transmissions where every field is identical.
2. **Coordinate Anomalies in GTFS `stops.txt`**:
   - Stop `2546` (*Libarty*): coordinates set to `(0.0, 0.0)`.
   - Stops `6501` (*Hanuman Mandir*) and `6725` (*Jain Mandir*): longitude was mistakenly duplicated from latitude (`lon == lat`).
3. **No Bad Coordinates or Extreme Speeds in GPS Telemetry**:
   - `0` zero coordinates and `0` out-of-bounds coordinates.
   - Max speed recorded is `5.833 m/s` (`21.0 km/h`); no impossible GPS multipath spikes ($>120\text{ km/h}$).
4. **Idling / Stationary Time**:
   - `35.36%` of all pings report $0.0\text{ m/s}$, reflecting heavy terminal wait times, traffic signal queues, and passenger dwell.
5. **Missing Values**:
   - `route_short_name` in `routes.txt` is 100% null (route names are under `route_long_name`).
   - `agency_phone` and `agency_fare_url` in `agency.txt` are 100% null.
